In [1]:

import os

os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"


import sys
sys.path.append("/home/79648j/Various/Autre/lepto/lepto/")

import numpy as np
import pandas as pd
from lepto.risk.model.linear_model import GLMDiff

# Your class import
# from yourmodule import GLMDiff  # Adapt import to your own project layout

# --- Create a synthetic dataset ---
rng = np.random.default_rng(42)

n = 20000

# Numeric features
df = pd.DataFrame({
    "age": rng.normal(40, 10, size=n),            # numeric
    "exposure": rng.uniform(0.5, 2.0, size=n),    # numeric
})

# Categorical features
regions = np.array(["north", "south", "east", "west"])
segments = np.array(["A", "B"])
df["region"] = rng.choice(regions, size=n)
df["segment"] = rng.choice(segments, size=n)

# Target (Poisson-like) with a simple generative structure
# True linear predictor (just for simulation)
lp = (
    0.1 * (df["age"] - 40) +         # effect of age centered
    0.5 * (df["segment"] == "B") +   # segment B higher
    0.2 * (df["region"] == "south")  # south slightly higher
)

mu = np.exp(lp) * df["exposure"]    # mean proportional to exposure
y = rng.poisson(mu) / df["exposure"]

# Optional sample weights (e.g., use exposure as weight)
sample_weight = df["exposure"].values

# Optional penalty matrix example for categorical variable "region"
# (e.g., neighboring regions more similar; diagonal 0, off-diagonals encode adjacency weights)
# Matrix must be k x k with k the number of categories for "region".
# Here we put a simple "ring" structure: north-east-south-west-north
region_index = {reg: i for i, reg in enumerate(regions)}
k = len(regions)
adj = np.zeros((k, k))
for i in range(k):
    adj[i, (i+1) % k] = 1.0
    adj[i, (i-1) % k] = 1.0
penalty_matrix = {"region": adj}

penalty = {'age': {'penalty':'continuous'}, 'region': {'penalty':'graph', 'graph':adj}}
# Optional offset betas: fix some coefficients and let others be optimized (np.nan).
# Example: If your total number of betas is known, you can pass an array with fixed/np.nan entries.
# For illustration only—leave None if not needed.
offset_betas = None

# --- Fit the GLM ---
glm = GLMDiff(
    family="poisson",        # default
    tweedie_power=1.5,       # only used for tweedie
    fit_intercept=True,
    nbins=10,                # fewer bins for the example
    lam=1e-3,
    penalty_choice=penalty,
    offset_betas=offset_betas
)

glm.fit(
    X=df[["age", "region", "segment"]],
    y=y,
    sample_weight=sample_weight
)

# --- Predict ---
pred = glm.predict(df[["age", "region", "segment"]])

# --- Summary (coefficients table) ---
glm.summary


{'intercept': np.float64(-1.6656502179682726),
 'coefficients': {'age': {'(27.0834, 31.6674]': np.float64(0.65285),
   '(31.6674, 34.7939]': np.float64(1.03912),
   '(34.7939, 37.5566]': np.float64(1.27688),
   '(37.5566, 39.9829]': np.float64(1.55992),
   '(39.9829, 42.523]': np.float64(1.80509),
   '(42.523, 45.25]': np.float64(2.07639),
   '(45.25, 48.457]': np.float64(2.36563),
   '(48.457, 53.0475]': np.float64(2.73384),
   '(53.0475, 81.5124]': np.float64(3.54995)},
  'region': {'north': np.float64(-0.01108),
   'south': np.float64(0.18276),
   'west': np.float64(0.00877)},
  'segment': {'B': np.float64(0.48831)}},
 'link': 'log'}

In [2]:
glm.plot(df[["age", "region", "segment"]], y, sample_weight, "age")

In [3]:
print(np.mean(pred))
print(np.mean(y))

2.3157213555275846
2.315721355527584


In [4]:
from sklearn.model_selection import GridSearchCV
parameters = {'lam':[1, 10]}
clf = GridSearchCV(glm, parameters)
clf.fit(X=df[["age", "region", "segment"]],
    y=y,
    sample_weight=sample_weight)
clf.best_params_

{'lam': 1}

In [5]:
# SPlit tran/test

In [6]:
from lepto.risk.model.transformers import GLMData

In [7]:
var_select = ["age", "region", "segment"]
X_raw = df[["age", "region", "segment"]]
transformer_data = GLMData()
transformer_data.fit(X_raw)
X_new = pd.DataFrame(transformer_data.transformer.named_steps['col_transformer'].transform(X_raw), columns=["age", "region", "segment"])
for var in transformer_data.var_num:
    categories_var = transformer_data._get_categories_var()
    X_new[var] = X_new[var].map({i: categories_var[var][i] for i in range(len(categories_var[var]))})
X_new

,age,region,segment
0,"(42.523, 43.8776]",east,B
1,"(27.0834, 29.6554]",east,B
2,"(46.7277, 48.457]",west,B
3,"(48.457, 50.5862]",east,A
4,"(-3.89115, 23.5622]",west,A
...,...,...,...
19995,"(56.7032, 81.5124]",west,A
19996,"(31.6674, 33.299]",west,A
19997,"(45.25, 46.7277]",west,A
19998,"(45.25, 46.7277]",east,A


In [27]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
def plot_distribution(variable, y, sample_weights, var_name="", nbins=20):
    
    """
    Plot Observed vs Predicted by category of `variable`, with Exposure on a secondary y-axis,
    using Plotly Express traces.

    Parameters
    ----------
    variable : array-like
    y : array-like
        Target on the response scale (e.g., frequency, severity, probability).
    sample_weights : array-like or None, optional
    var_name : str
    nbins : int

    Returns
    -------
    fig : plotly.graph_objects.Figure
        Figure with Observed/Predicted lines and Exposure bars (secondary y).
    """

    if sample_weights is None:
        sample_weights = np.ones_like(y, dtype=float)
    else:
        sample_weights = np.array(sample_weights, dtype=float)

    # Cut numeric into bins
    if pd.api.types.is_numeric_dtype(variable):
         new_var = pd.cut(variable, nbins)
         categories = new_var.cat.categories.astype(str)
         mapping = {cat: f"{i:03} : {cat}" for i, cat in enumerate(categories, start=1)}
         variable = new_var.astype(str).map(mapping)

    # --- Inputs normalization ---
    variable = np.array(variable)
    # --- Build working frame ---
    df = pd.DataFrame(
        {
            "var": variable,
            "y": y,
            "w": sample_weights}
    )
    df["y_w"] = df["y"] * df["w"]

    # --- Aggregations by category ---
    # exposure = sum of weights
    gb_expo = df.groupby("var", observed=False)["w"].sum().rename("exposure")
    # observed = sum(y * w) / sum(w)
    gb_obs = (df.groupby("var", observed=False)["y_w"].sum() / gb_expo).rename("observed")
    df_gb = pd.concat([gb_obs, gb_expo], axis=1).reset_index()


    # Create subplot with secondary y-axis
    fig = make_subplots()

    # Exposure bars
    fig.add_trace(go.Bar(
        x=df_gb["var"], y=df_gb["exposure"],
        name="Exposure",
        opacity=0.5
    ))

   
    # Layout
    fig.update_layout(
        title=f"Distribution: {var_name}" if var_name else "Distribution",
        xaxis=dict(title=var_name or "Variable", type="category"),
        yaxis=dict(title="Value"),
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
    )

    return fig

In [28]:
plot_distribution(df['age'], y, sample_weight, var_name="")

In [8]:
from lepto.risk.model.linear_model import analyse_var
var_an = 'age'
analyse_var(variable=X_new[var_an],
            y=y,
            sample_weights=sample_weight,
                                   preds=None,
                                   coef=None,
                                   var_name=var_an)

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances
from lepto.risk.model.penalty import create_graph_continuous, create_graph_categorical


def create_graph_geographical(geographical_data):
    # Create pairwise distance
    graph = pairwise_distances(geographical_data[['lat']].to_numpy(), geographical_data[['lon']].to_numpy(), metric='euclidean')
    return graph


In [19]:
def create_graph_matrix(var_type, geographical_data=None):
    adj_dict = {}
    for var in var_select :
        if var_type[var] == 'continuous':
            adj_dict[var] = create_graph_continuous(len(categories_var[var]))
        elif var_type[var] == 'categorical':
            adj_dict[var] = create_graph_categorical(len(categories_var[var]))
        elif var_type[var] == 'geographical':
            adj_dict[var] = create_graph_geographical(geographical_data[var])
    return adj_dict

In [20]:
# Create difference matrix
geographical_data_dict = {}
geographical_data_dict["region"] = pd.DataFrame({"region": ['east', 'west', 'south', 'north', 'new_zone'], "lat": [1, -1, 0, 0, 2], "lon": [0, 0, -1, 1, 2]})
var_type = {"age": "continuous", "region": "geographical", "segment": "categorical"}
adj_dict = create_graph_matrix(var_type, geographical_data_dict)

In [22]:
def create_categories(transformer, geographical_data):
    # Create categories
    categories_list = transformer.transformer.named_steps['onehot'].categories_
    for i, var in enumerate(transformer.var_num + transformer.var_cat):
        if var_type[var] == 'geographical':
            categories_list[i] = geographical_data[var][var].unique()
    return categories_list

categories_list = create_categories(transformer_data, geographical_data_dict)

In [23]:
def create_offset(transformer, categories_list):
    # Create offset
    offset_betas = {}
    for i, var in enumerate(transformer.var_num + transformer.var_cat) :
        offset_betas[var] = np.full(len(categories_list[i])-1, np.nan)
    return offset_betas

offset_betas = create_offset(transformer_data, categories_list)
full_offset = np.concatenate(list(offset_betas.values()))
full_offset = np.append(full_offset, np.nan)

In [24]:
full_offset

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])

In [ ]:
# Build framework
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import GridSearchCV

class GLMFramework(BaseEstimator, RegressorMixin):
    def __init__(self,
                 family="poisson",
                 tweedie_power = 1.5,
                 lam_grid=np.linspace(1e-6, 10, 50),
                 adj_matrix=None,
                 offset_betas=None,
                 categories='auto'):
          
          self.fit_intercept = True
          self.nbins = 20
          self.family = family 
          self.tweedie_power = tweedie_power
          self.lam_grid = lam_grid
          self.adj_matrix = adj_matrix
          self.offset_betas = offset_betas
          self.categories = categories
          self.grid_search = None

    def fit(self,
            X,
            y,
            sample_weight=None):
        
        # init GLM
        ## Offset
        full_offset = np.concatenate(list(self.offset_betas.values()))
        full_offset = np.append(full_offset, np.nan)
        ## Penalty
        penalty = {}
        for var in self.adj_matrix:
            penalty[var] = {'penalty':'graph', 'graph':self.adj_matrix[var]}
        
        # GLM
        glm = GLMDiff(
                family=self.family ,      
                tweedie_power=self.tweedie_power,      
                fit_intercept=True,
                nbins=self.nbins,               
                lam=1e-3,
                penalty_choice=penalty,
                offset_betas=full_offset,
                categories = self.categories
            )
        ## Grid serach CV --> best
        parameters = {'lam': self.lam_grid}
        self.grid_search = GridSearchCV(glm, parameters, refit=True)
        self.grid_search.fit(X=X,
            y=y,
            sample_weight=sample_weight)
        
        # Store model
        self.estimator = self.grid_search.best_estimator_
        self.is_fitted_ = True


    def predict(self, X):

        self.estimator.predict(X)

    def refit(self, lam, penalty_choice, offset_betas):
        # GLM
        glm = GLMDiff(
                family=self.family ,      
                tweedie_power=self.tweedie_power,      
                fit_intercept=True,
                nbins=self.nbins,               
                lam=lam,
                penalty_choice=penalty_choice,
                offset_betas=offset_betas,
                categories = self.categories
            )
        self.estimator = glm

    def rebase(self):
        # GLM
        penalty_choice = self.estimator.penalty_choice
        offset_betas = self.estimator.pipeline.named_steps['model'].betas
        offset_betas[-1] = np.nan
        glm = GLMDiff(
                family=self.family ,      
                tweedie_power=self.tweedie_power,      
                fit_intercept=True,
                nbins=self.nbins,               
                lam=self.estimator.lam,
                penalty_choice=penalty_choice,
                offset_betas=offset_betas,
                categories = self.categories
            )
        self.estimator = glm





In [15]:
glm_test = GLMFramework(family="poisson",
                 tweedie_power = 1.5,
                 lam_grid=np.linspace(1e-6, 10, 2),
                 adj_matrix=adj_dict,
                 offset_betas=offset_betas,
                 categories = categories_list)
glm_test.fit(X_raw, y, sample_weight)

In [17]:
glm_test.grid_search.best_estimator_.summary

{'intercept': np.float64(-2.0613015325345647),
 'coefficients': {'age': {'(23.5622, 27.0834]': np.float64(0.66302),
   '(27.0834, 29.6554]': np.float64(0.91104),
   '(29.6554, 31.6674]': np.float64(1.14015),
   '(31.6674, 33.299]': np.float64(1.31521),
   '(33.299, 34.7939]': np.float64(1.51683),
   '(34.7939, 36.2525]': np.float64(1.57317),
   '(36.2525, 37.5566]': np.float64(1.73394),
   '(37.5566, 38.7732]': np.float64(1.9211),
   '(38.7732, 39.9829]': np.float64(1.96017),
   '(39.9829, 41.2465]': np.float64(2.13671),
   '(41.2465, 42.523]': np.float64(2.23503),
   '(42.523, 43.8776]': np.float64(2.37735),
   '(43.8776, 45.25]': np.float64(2.52988),
   '(45.25, 46.7277]': np.float64(2.66613),
   '(46.7277, 48.457]': np.float64(2.82389),
   '(48.457, 50.5862]': np.float64(2.99932),
   '(50.5862, 53.0475]': np.float64(3.21991),
   '(53.0475, 56.7032]': np.float64(3.53965),
   '(56.7032, 81.5124]': np.float64(4.22652)},
  'region': {'west': np.float64(0.01226),
   'south': np.float64(0